In [0]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

# Preliminary Version, must add batch process and filter results to philippines only before merging, fix the snapshot date to the actual snapshot date or maybe remove the snapshot date
spark = SparkSession.builder.appName("Bronze").getOrCreate()

metadata_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .option("sep", "\t") \
    .load("s3a://inaturalist-open-data/observations.csv.gz")

ingested_df = metadata_df.withColumn(
    "ingestion_timestamp",
    F.current_timestamp()
).withColumn(
    "source_file_path",
    F.col("_metadata.file_path")
).withColumn(
    "snapshot_date",
    F.current_date()
)

ingested_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("philippinebiodiveristyplatform.bronze.observations")